# Imputation Environment Check

Run the next cell to verify that required libraries are available for:
- Mean imputation (`SimpleImputer`)
- kNN imputation (`KNNImputer`)
- MICE and SoftImpute workflows (via `hyperimpute`)
- Optional research repos (`GRAPE`, `DiffPuter`)

In [5]:
import sys
import pathlib
import importlib
from importlib import metadata

# ---- self-contained DiffPuter path setup ----------------------------------
# This makes the cell work on its own. If you also have a separate path-setup
# cell, that's fine — running it twice is a no-op.
HERE = pathlib.Path.cwd()
for cand in [
    HERE / "DiffPuter",
    HERE / "external" / "DiffPuter",
    HERE.parent / "DiffPuter",
]:
    if (cand / "main.py").is_file():
        for p in [cand, cand / "baselines", cand / "baselines" / "GRAPE"]:
            sp = str(p.resolve())
            if sp not in sys.path:
                sys.path.insert(0, sp)
        break

# Module name (for import) -> distribution name (for version lookup).
# These differ for some packages (sklearn vs scikit-learn, yaml vs PyYAML,
# ot vs POT).
required_modules = {
    # core scientific
    "numpy":         "numpy",
    "pandas":        "pandas",
    "scipy":         "scipy",
    "sklearn":       "scikit-learn",
    "matplotlib":    "matplotlib",
    "statsmodels":   "statsmodels",

    # baselines + SOTA
    "fancyimpute":   "fancyimpute",     # Mean (SimpleFill), kNN, SoftImpute
    "hyperimpute":   "hyperimpute",     # HyperImpute + MICE plugin

    # deep learning + GRAPE deps
    "torch":             "torch",
    "torch_geometric":   "torch_geometric",
    "h5py":              "h5py",
    "networkx":          "networkx",

    # DiffPuter deps
    "ot":            "POT",             # POT distributes as POT, imports as ot
    "FrEIA":         "FrEIA",
    "timm":          "timm",
    "yaml":          "PyYAML",

    # notebook
    "ipykernel":     "ipykernel",
}

# Modules that are nice-to-have but the env still works without them.
optional_modules = {
    "torch_scatter": "torch_scatter",   # GRAPE only
}

# Note: GRAPE and DiffPuter are loaded as local source clones, not pip packages.


def check_module(module_name: str, package_name: str):
    try:
        importlib.import_module(module_name)
    except Exception as e:
        return "MISSING", str(e)
    try:
        version = metadata.version(package_name)
    except metadata.PackageNotFoundError:
        version = "unknown"
    return "OK", version


print("Required packages")
print("-" * 60)
missing = []
for module_name, package_name in required_modules.items():
    status, info = check_module(module_name, package_name)
    marker = "✓" if status == "OK" else "✗"
    print(f"  {marker} {package_name:18} {status:8} {info}")
    if status == "MISSING":
        missing.append(package_name)

print("\nOptional packages")
print("-" * 60)
for module_name, package_name in optional_modules.items():
    status, info = check_module(module_name, package_name)
    marker = "✓" if status == "OK" else "○"
    print(f"  {marker} {package_name:18} {status:8} {info}")

print("\nSmoke-test imports for imputation APIs")
print("-" * 60)

api_checks = [
    ("Mean / kNN (sklearn)",
     lambda: __import__("sklearn.impute", fromlist=["SimpleImputer", "KNNImputer"])),
    ("MICE (sklearn IterativeImputer)",
     lambda: (
         __import__("sklearn.experimental", fromlist=["enable_iterative_imputer"]),
         __import__("sklearn.impute",       fromlist=["IterativeImputer"]),
     )),
    ("SoftImpute / KNN / SimpleFill (fancyimpute)",
     lambda: __import__("fancyimpute", fromlist=["SoftImpute", "KNN", "SimpleFill"])),
    ("HyperImpute (hyperimpute.plugins.imputers.Imputers)",
     lambda: __import__("hyperimpute.plugins.imputers", fromlist=["Imputers"])),
    ("PyTorch Geometric (for GRAPE)",
     lambda: __import__("torch_geometric")),
]

for label, fn in api_checks:
    try:
        fn()
        print(f"  ✓ {label}: OK")
    except Exception as e:
        print(f"  ✗ {label}: FAILED | {e}")

print("\nLocal repo modules (DiffPuter / GRAPE)")
print("-" * 60)
repo_checks = [
    ("DiffPuter dataset",   "dataset",          ["load_dataset", "get_eval", "mean_std"]),
    ("DiffPuter diffusion", "diffusion_utils",  ["sample_step", "impute_mask", "EDMLoss"]),
    ("DiffPuter model",     "model",            ["MLPDiffusion", "Model"]),
    ("GRAPE training",      "training.gnn_mdi", ["train_gnn_mdi"]),
]
for label, mod, names in repo_checks:
    try:
        m = importlib.import_module(mod)
        for n in names:
            getattr(m, n)
        print(f"  ✓ {label}: OK ({mod})")
    except Exception as e:
        print(f"  ✗ {label}: FAILED | {type(e).__name__}: {e}")

print()
if missing:
    print(f"⚠  Missing required packages: {', '.join(missing)}")
    print("   Re-run: pip install -r requirements-base.txt")
else:
    print("All required packages present.")

Required packages
------------------------------------------------------------
  ✓ numpy              OK       1.26.4
  ✓ pandas             OK       3.0.2
  ✓ scipy              OK       1.17.1
  ✓ scikit-learn       OK       1.8.0
  ✓ matplotlib         OK       3.10.9
  ✓ statsmodels        OK       0.14.6
  ✓ fancyimpute        OK       0.7.0
  ✓ hyperimpute        OK       0.1.17
  ✓ torch              OK       2.11.0
  ✓ torch_geometric    OK       2.7.0
  ✓ h5py               OK       3.14.0
  ✓ networkx           OK       3.6.1


I0000 00:00:1777906925.671012    2893 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1777906936.458325    2893 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


  ✓ POT                OK       0.9.6.post1
  ✓ FrEIA              OK       0.2
  ✓ timm               OK       1.0.26
  ✓ PyYAML             OK       6.0.3
  ✓ ipykernel          OK       7.2.0

Optional packages
------------------------------------------------------------
  ✓ torch_scatter      OK       2.1.2+pt211cu130

Smoke-test imports for imputation APIs
------------------------------------------------------------
  ✓ Mean / kNN (sklearn): OK
  ✓ MICE (sklearn IterativeImputer): OK
  ✓ SoftImpute / KNN / SimpleFill (fancyimpute): OK
  ✓ HyperImpute (hyperimpute.plugins.imputers.Imputers): OK
  ✓ PyTorch Geometric (for GRAPE): OK

Local repo modules (DiffPuter / GRAPE)
------------------------------------------------------------
  ✓ DiffPuter dataset: OK (dataset)
  ✓ DiffPuter diffusion: OK (diffusion_utils)
  ✓ DiffPuter model: OK (model)
  ✓ GRAPE training: OK (training.gnn_mdi)

All required packages present.
